# Reproduce the Apple/HarmonyOS usability results (Tables 2 and 3, human validation, sensitivity)

Recomputes every Apple and HarmonyOS usability number reported in the paper from the released files only:
no API calls, no raw post text. Runs in Google Colab or locally in about a minute.

Set `BASE` to the `Apple_HarmonyOS_Analysis` folder of the replication package.

In [ ]:
import os, re, json
import numpy as np, pandas as pd
from sklearn.metrics import cohen_kappa_score
BASE = "."
idx  = pd.read_csv(f"{BASE}/data/posts_index_v4.csv")
allp = pd.read_csv(f"{BASE}/data/extracted_aspect_sentiment_v4_all.csv")
ver  = pd.read_csv(f"{BASE}/data/extracted_aspect_sentiment_v4_verified.csv")
cons = pd.read_csv(f"{BASE}/human_validation/usability_consensus_labels_v4.csv")
print(len(idx), "posts |", len(allp), "first-pass pairs |", len(ver), "verified pairs |", len(cons), "hand-labeled posts")


## Helpers: cluster bootstrap and concentration

In [ ]:
def boot_ci(pol, n=5000, seed=0):
    gs = [g["pos"].values for _, g in pol.groupby("cluster")]
    rng = np.random.default_rng(seed)
    b = [np.concatenate([gs[i] for i in rng.integers(0, len(gs), len(gs))]).mean() for _ in range(n)]
    return tuple(round(float(x), 1) for x in np.percentile(b, [2.5, 97.5]) * 100)

def polar(pairs):
    return pairs[pairs.sentiment.isin(["positive", "negative"])].assign(pos=lambda d: (d.sentiment == "positive").astype(int))

def summarize(pairs, name):
    posts = pairs.drop_duplicates("url"); per = posts.groupby("cluster").size().sort_values(ascending=False)
    p = polar(pairs); lo, hi = boot_ci(p)
    return {"platform": name, "posts": len(posts), "clusters": per.size,
            "eff_clusters": round(per.sum() ** 2 / (per ** 2).sum(), 1), "top1_pct": round(100 * per.iloc[0] / per.sum(), 1),
            "pairs": len(pairs), "positive_share_pct": round(100 * p.pos.mean(), 1), "ci_low": lo, "ci_high": hi,
            "reportable (>=30 clusters, top1<=20%)": per.size >= 30 and per.iloc[0] / per.sum() <= 0.20}


## Table 2: final estimates
HarmonyOS: posts confirmed by both authors' reconciled labels (first-pass aspect pairs).
Apple: posts retained by the precision pass and detected as English (verified aspect pairs).

In [ ]:
final = idx[idx.analyzed_final]
H = allp[(allp.platform == "harmonyos") & allp.url.isin(final[final.platform == "harmonyos"].url)]
A = ver[(ver.platform == "apple") & ver.url.isin(final[final.platform == "apple"].url)]
table2 = pd.DataFrame([summarize(H, "HarmonyOS"), summarize(A, "Apple/iOS")])
print(table2.to_string(index=False))
print("\nPaper: HarmonyOS 302 posts, 146 clusters, 62.6 eff., 396 pairs, 44.8% [36.4, 53.5];"
      " Apple 9,003 posts, 3,452 clusters, 899.8 eff., 15,860 pairs, 23.4% [22.0, 24.9]")
print("Windows (64.8% [64.1, 65.6]) requires the Chhetri et al. corpus; see ../Copilot_Quantitative_Analysis.")


## Table 3: posts by source (collected, clusters, retained by the precision pass, analyzed)

In [ ]:
t3 = idx.groupby(["platform", "source"]).agg(posts=("url", "size"), clusters=("cluster", "nunique"),
                                            retained=("verifier_keep", "sum"), analyzed=("analyzed_final", "sum"))
print(t3.to_string())


## Human validation: inter-rater agreement and model vs consensus

In [ ]:
k = lambda a, b: round(cohen_kappa_score(list(a), list(b)), 3)
both_y = cons[(cons.A_rel == "Y") & (cons.B_rel == "Y")].dropna(subset=["A_sent", "B_sent"])
print(f"Inter-rater, relevance: {100*(cons.A_rel == cons.B_rel).mean():.1f}% agreement, kappa = {k(cons.A_rel, cons.B_rel)}")
print(f"Inter-rater, sentiment (both Y, n={len(both_y)}): {100*(both_y.A_sent == both_y.B_sent).mean():.1f}%, "
      f"kappa = {k(both_y.A_sent, both_y.B_sent)}")
rows = []
for plat, g in [("all", cons)] + list(cons.groupby("platform")):
    tp = ((g.model_rel == "Y") & (g.H_rel == "Y")).sum(); fp = ((g.model_rel == "Y") & (g.H_rel == "N")).sum()
    fn = ((g.model_rel == "N") & (g.H_rel == "Y")).sum()
    yy = g[(g.model_rel == "Y") & (g.H_rel == "Y")].dropna(subset=["H_sent"])
    rows.append({"platform": plat, "precision": round(tp / (tp + fp), 3), "recall": round(tp / (tp + fn), 3),
                 "relevance_kappa": k(g.model_rel, g.H_rel),
                 "sentiment_agreement_pct": round(100 * (yy.model_sent == yy.H_sent).mean(), 1),
                 "sentiment_kappa": k(yy.model_sent, yy.H_sent)})
print(pd.DataFrame(rows).to_string(index=False))
print("Paper: inter-rater kappa 0.896 / 0.934; model precision 0.653 (Apple 0.805, HarmonyOS 0.622), recall 0.834,"
      " relevance kappa 0.372; sentiment 88.4%, kappa 0.797")


## Human-label sentiment estimates and the superseded model-only HarmonyOS estimate

In [ ]:
hs = cons[(cons.platform == "harmonyos") & (cons.H_rel == "Y") & cons.H_sent.isin(["positive", "negative"])].copy()
hs = hs.merge(idx[["platform", "url", "cluster"]], on=["platform", "url"]); hs["pos"] = (hs.H_sent == "positive").astype(int)
print(f"HarmonyOS, authors' own sentiment (post level): {100*hs.pos.mean():.1f}% {boot_ci(hs)}   [paper: 46.2% (37.0, 55.5)]")
ah = cons[(cons.platform == "apple") & (cons.H_rel == "Y") & cons.H_sent.isin(["positive", "negative"])].copy()
ah = ah.merge(idx[["platform", "url", "cluster"]], on=["platform", "url"]); ah["pos"] = (ah.H_sent == "positive").astype(int)
print(f"Apple, human-validated sample (n={len(ah)}): {100*ah.pos.mean():.1f}% {boot_ci(ah)}   [paper: 27.0% (16.1, 38.7)]")
Hm = ver[ver.platform == "harmonyos"]; pm = polar(Hm)
print(f"HarmonyOS, model relevance only (superseded): {100*pm.pos.mean():.1f}% {boot_ci(pm)}   [paper: 52.6% (44.5, 60.8)]")


## Sensitivity: cap of 10 posts per cluster, and excluding camera/photo AI

In [ ]:
CAM = re.compile(r"camera|photo|portrait|zoom|pose|selfie|lens|snapshot|picture|night ?mode|low ?light|xmage|shot|clarity|blur|oversharpen|exposure|image quality", re.I)
NOTCAM = re.compile(r"generat|playground|genmoji|emoji", re.I)
is_cam = lambda s: bool(CAM.search(str(s))) and not NOTCAM.search(str(s))
for name, P in [("HarmonyOS", H), ("Apple/iOS", A)]:
    posts = P.drop_duplicates("url")
    keep = pd.concat([g.sample(min(len(g), 10), random_state=0) for _, g in posts.groupby("cluster")]).url
    cap = polar(P[P.url.isin(keep)]).pos.mean() * 100
    nc = polar(P[~P.summary.map(is_cam)]); cam = polar(P[P.summary.map(is_cam)])
    print(f"{name}: cap10 {cap:.1f}% | camera/photo share of pairs {100*P.summary.map(is_cam).mean():.1f}% | "
          f"excluding camera {100*nc.pos.mean():.1f}% {boot_ci(nc)} | camera only {100*cam.pos.mean():.1f}%")
print("Paper: HarmonyOS cap10 45.3, camera 21.5%, excl. 43.2 (34.3, 52.5), camera only 50.6;"
      " Apple cap10 23.4, camera 1.3%, excl. 23.2 (21.8, 24.7), camera only 40.9")
